# AeroSense — Air-Quality Forecasting (methodology notebook)

This notebook mirrors the from-scratch JavaScript pipeline (`ml/train.mjs`) using the standard
Python data-science stack (`pandas` + `scikit-learn`). It documents the **same methodology**:
feature engineering, a chronological train/test split, ridge regression, and evaluation against
naïve baselines.

**Dataset:** UCI Beijing PM2.5 (`../data/beijing_pm25.csv`).

> Run with: `pip install pandas scikit-learn numpy` then execute top to bottom.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (r2_score, mean_absolute_error, mean_squared_error,
                             accuracy_score, f1_score, confusion_matrix)

## 1. Load the data

In [ ]:
df = pd.read_csv('../data/beijing_pm25.csv').rename(columns={'pm2.5': 'pm25'})
print('raw shape:', df.shape)
df.head()

## 2. Feature engineering

Autoregressive lags + rolling mean, cyclical time encodings, and one-hot wind direction —
identical to `ml/lib/preprocess.mjs`.

In [ ]:
# US EPA PM2.5 -> AQI category (0..5)
BINS = [0, 12.05, 35.45, 55.45, 150.45, 250.45, np.inf]
def to_category(pm):
    return np.clip(np.digitize(pm, BINS) - 1, 0, 5)

d = df.copy()
for k in (1, 2, 3):
    d['pm25_lag%d' % k] = d['pm25'].shift(k)
d['pm25_roll3'] = d[['pm25_lag1', 'pm25_lag2', 'pm25_lag3']].mean(axis=1)
d['hour_sin'] = np.sin(2 * np.pi * d['hour'] / 24)
d['hour_cos'] = np.cos(2 * np.pi * d['hour'] / 24)
d['month_sin'] = np.sin(2 * np.pi * (d['month'] - 1) / 12)
d['month_cos'] = np.cos(2 * np.pi * (d['month'] - 1) / 12)
for w in ('NW', 'NE', 'SE'):
    d['wind_%s' % w] = (d['cbwd'] == w).astype(int)

d = d.dropna(subset=['pm25', 'pm25_lag1', 'pm25_lag2', 'pm25_lag3']).reset_index(drop=True)
d['y_cat'] = to_category(d['pm25'].values)
print('usable rows:', len(d))

## 3. Chronological split (no temporal leakage)

First 80% (2010–2013) trains, last 20% (2014) tests. Standardisation is fit on train only.

In [ ]:
FORECAST = ['pm25_lag1', 'pm25_lag2', 'pm25_lag3', 'pm25_roll3', 'DEWP', 'TEMP', 'PRES',
            'Iws', 'Is', 'Ir', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos',
            'wind_NW', 'wind_NE', 'wind_SE']
SCENARIO = FORECAST[4:]  # weather + time only

cut = int(len(d) * 0.8)
train, test = d.iloc[:cut], d.iloc[cut:]
ytr, yte = train['pm25'].values, test['pm25'].values

def prep(cols):
    sc = StandardScaler().fit(train[cols])
    return sc.transform(train[cols]), sc.transform(test[cols])

Xtr_f, Xte_f = prep(FORECAST)
Xtr_s, Xte_s = prep(SCENARIO)
print('train/test:', len(train), len(test))

## 4. Train ridge regressors

In [ ]:
forecast = Ridge(alpha=5.0).fit(Xtr_f, ytr)
scenario = Ridge(alpha=5.0).fit(Xtr_s, ytr)
pred_f = np.clip(forecast.predict(Xte_f), 0, None)
pred_s = np.clip(scenario.predict(Xte_s), 0, None)

## 5. Evaluate (regression + derived AQI category)

In [ ]:
def report(name, pred):
    ct, cp = test['y_cat'].values, to_category(pred)
    r2 = r2_score(yte, pred)
    mae = mean_absolute_error(yte, pred)
    rmse = mean_squared_error(yte, pred) ** 0.5
    acc = accuracy_score(ct, cp)
    mf1 = f1_score(ct, cp, average='macro')
    print(name, 'R2=%.3f MAE=%.1f RMSE=%.1f acc=%.3f macroF1=%.3f' % (r2, mae, rmse, acc, mf1))

report('forecast', pred_f)
report('scenario', pred_s)

## 6. Naïve baselines & confusion matrix

In [ ]:
persist = test['pm25_lag1'].values  # carry the previous hour forward
print('persistence  R2=%.3f acc=%.3f' % (
    r2_score(yte, persist),
    accuracy_score(test['y_cat'].values, to_category(persist))))
print('\nconfusion matrix (forecast model):')
print(confusion_matrix(test['y_cat'].values, to_category(pred_f)))

## Summary

The forecast ridge model reproduces the result shipped in the app (**R² ≈ 0.95, MAE ≈ 12 µg/m³**),
confirming the dependency-free JavaScript pipeline in `ml/` is correct. The exported weights in
`public/model.json` then run client-side in the browser with identical preprocessing.